In [2]:
#load apikeys from .env

from dotenv import load_dotenv
import os
import getpass

load_dotenv()


if "ANTHROPIC_API_KEY" not in os.environ:
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Introduce tu  AWS Key: ")

In [3]:
#model agnostic function for llm inference with init_chat_model and invoke()

from langchain.chat_models import init_chat_model

def generate_example_orders(model_name, prompt, provider=None, token_limit=200, temperature = 0.7):
    model = init_chat_model(
        model_name,
        model_provider=provider,
        max_tokens=token_limit,
        temperature = temperature
    )
    response = model.invoke(prompt)
    return response

In [4]:
model_name = "claude-haiku-4-5-20251001"
provider = "anthropic"
prompt = "Generate the transcription of a demanding customer ordering a pizza with a severe nut allergy, who interrogates the staff about cross-contamination in the kitchen."


response = generate_example_orders(
    model_name,
    prompt,
    provider
)

response

AIMessage(content='# Pizza Order Transcription\n\n**Customer:** Hi, yes, I need to place an order, but I have some very specific requirements. I have a severe nut allergy—anaphylactic—so I need to ask you some detailed questions before I order.\n\n**Staff Member:** Of course. I can help you with that. Let me get some information.\n\n**Customer:** First, do you have any nut products in this kitchen at all? Any pesto with pine nuts, almond flour, anything?\n\n**Staff:** We do have some items with nuts. We have a pesto sauce and a few specialty items.\n\n**Customer:** Okay, so you do handle nuts. That\'s what I needed to know. So tell me—how do you prevent cross-contamination? Do you have separate prep areas?\n\n**Staff:** We have different stations, and we can prepare nut-free items separately.\n\n**Customer:** "Can', additional_kwargs={}, response_metadata={'id': 'msg_011Cf58u5UHRSHwNpbup6mvR', 'container': None, 'model': 'claude-haiku-4-5-20251001', 'stop_details': None, 'stop_reason':

In [4]:
#Components of the AI Message

#response.content
#response.response_metadata
#response.tool_calls
#response.invalid_tool_calls
response.usage_metadata

{'input_tokens': 15,
 'output_tokens': 200,
 'total_tokens': 215,
 'input_token_details': {'cache_read': 0,
  'cache_creation': 0,
  'ephemeral_5m_input_tokens': 0,
  'ephemeral_1h_input_tokens': 0}}

In [6]:
#print out all messages

import json

def print_aimessage(ai_message):
    for key, value in ai_message.model_dump().items():                #convert Ai-Message to Python dictionary and get items
        if isinstance(value, (dict, list)):                         #check if the value is a nested structure
            print(f"\n[{key}]:")                                    
            print(json.dumps(value, indent=2, ensure_ascii=False))  #converts a Python object into a JSON-formatted string. It’s commonly used to serialize data for storage or sending
        else:
            print(f"\n[{key}]: {value}")

print_aimessage(response)


[content]: # Pizza Order Transcription

**Customer:** Hi, yes, I need to place an order, but I have some very specific requirements. I have a severe nut allergy—anaphylactic—so I need to ask you some detailed questions before I order.

**Staff Member:** Of course. I can help you with that. Let me get some information.

**Customer:** First, do you have any nut products in this kitchen at all? Any pesto with pine nuts, almond flour, anything?

**Staff:** We do have some items with nuts. We have a pesto sauce and a few specialty items.

**Customer:** Okay, so you do handle nuts. That's what I needed to know. So tell me—how do you prevent cross-contamination? Do you have separate prep areas?

**Staff:** We have different stations, and we can prepare nut-free items separately.

**Customer:** "Can

[additional_kwargs]:
{}

[response_metadata]:
{
  "id": "msg_011Cf58u5UHRSHwNpbup6mvR",
  "container": null,
  "model": "claude-haiku-4-5-20251001",
  "stop_details": null,
  "stop_reason": "max_

In [7]:
#start the ollama server
#uv add langchain_ollama
# utliza un modelo descargado

model_name = "qwen3:4b-instruct"
provider = "ollama"
prompt = "Generate the transcription of an indecisive customer who changes their pizza order three separate times before finally deciding."

response = generate_example_orders(
    model_name,
    prompt,
    provider
)

print_aimessage(response)


[content]: [Recording begins – ambient background noise of a restaurant, soft music playing in the background]

Customer (voice hesitant, slightly nervous):  
Uh… I’m not sure what I want. I mean, I’ve looked at the menu, and there’s so many options… I don’t know. I think I’ll go with the classic pepperoni, but I don’t know if I really like pepperoni. I mean, I’ve had it before, but it was kind of… too strong, right?  

Wait—no, maybe I don’t want pepperoni at all. What about the veggie one? I’ve seen it. It has mushrooms, olives, bell peppers, right? That sounds… nice. But I don’t know if I can eat a lot of vegetables. I’m not sure if I’ll actually enjoy it.  

Oh! What about the supreme? That one has pepperoni, sausage, and bacon, right? That sounds like a lot of meat. I don’t know if I can handle that. I don’t want to be full before the meal even starts.  

Wait—what if I just… go back to the classic? I think I might actually like the classic. It’s simple. It’s just cheese and pepp

In [20]:
import time
import pandas as pd
from pathlib import Path
from langchain.chat_models import init_chat_model

# 1. Define model configurations
models_config = [
    # Ollama Models
    {"model": "gemma3:4b", "provider": "ollama"},
    {"model": "gemma4:e4b", "provider": "ollama"},

    {"model": "llama3.1:8b", "provider": "ollama"},

    {"model": "ministral-3:3b", "provider": "ollama"},
    {"model": "mistral:7b", "provider": "ollama"},
    {"model": "mistral-nemo:12b", "provider": "ollama"},

    {"model": "qwen3.5:0.8b", "provider": "ollama"},
    {"model": "qwen3:4b-instruct", "provider": "ollama"},
    {"model": "qwen3:8b", "provider": "ollama"},
    {"model": "qwen3.5:9b", "provider": "ollama"},
    
    # Anthropic Model
    {"model": "claude-haiku-4-5-20251001", "provider": "anthropic"},
]

prompt = "Generate the transcription of two customers on the same call disagreeing about toppings and asking the staff to split the pizza order into custom halves and quarters.."
results = []



# 2. Invoke models and record metadata
for config in models_config:
    model_name = config["model"]
    provider = config["provider"]
    temperature = 1.0
    
    try:
        # Initialize LLM via LangChain init_chat_model
        llm = init_chat_model(model_name, model_provider=provider, temperature=temperature)
        
        # Track wall-clock duration
        start_time = time.perf_counter()
        response = llm.invoke(prompt)
        end_time = time.perf_counter()
        
        duration = round(end_time - start_time, 3)
        answer = response.content
        
        # Extract usage metadata
        usage_metadata = getattr(response, "usage_metadata", {}) or {}
        input_tokens = usage_metadata.get("input_tokens", None)
        output_tokens = usage_metadata.get("output_tokens", None)
        
        
        results.append({
            "modelname": model_name,
            "provider": provider,
            "duration_sec": duration,
            "input_tokens": input_tokens,
            "output_tokens": output_tokens,
            "answer": answer
        })
        print(f"Executed {model_name} in {duration} seconds. Input tokens: {input_tokens}, Output tokens: {output_tokens}, Answer: {answer[:80]}...")  # Print first 50 chars of answer for brevity
        
    except Exception as e:
        print(f"Error executing {model_name}: {e}")

# 3. Create Pandas DataFrame
df = pd.DataFrame(results)

df

Executed gemma3:4b in 16.916 seconds. Input tokens: 39, Output tokens: 983, Answer: Okay, here's a transcription of a call between two customers, Mark and Sarah, ar...
Error executing gemma4:e4b: model 'gemma4:e4b' not found (status code: 404)
Executed llama3.1:8b in 21.344 seconds. Input tokens: 39, Output tokens: 524, Answer: Here is the transcription of the call:

**Ringing...**

**Operator**: Thank you ...
Error executing ministral-3:3b: model 'ministral-3:3b' not found (status code: 404)
Executed mistral:7b in 14.545 seconds. Input tokens: 37, Output tokens: 290, Answer:  Customer 1: Hi, I'd like to place an order for a large pizza, but I'd like it w...
Error executing mistral-nemo:12b: model 'mistral-nemo:12b' not found (status code: 404)
Error executing qwen3.5:0.8b: model 'qwen3.5:0.8b' not found (status code: 404)
Executed qwen3:4b-instruct in 22.022 seconds. Input tokens: 37, Output tokens: 1140, Answer: [Phone call transcription – Customer Service Representative (CSR) and tw

,modelname,provider,duration_sec,input_tokens,output_tokens,answer
0,gemma3:4b,ollama,16.916,39,983,"Okay, here's a transcription of a call between..."
1,llama3.1:8b,ollama,21.344,39,524,Here is the transcription of the call:\n\n**Ri...
2,mistral:7b,ollama,14.545,37,290,"Customer 1: Hi, I'd like to place an order fo..."
3,qwen3:4b-instruct,ollama,22.022,37,1140,[Phone call transcription – Customer Service R...
4,qwen3:8b,ollama,27.496,39,999,**Transcription of Customer Call:** \n\n**[Ph...
5,claude-haiku-4-5-20251001,anthropic,9.022,40,782,# Pizza Order Call Transcription\n\n**[Call be...


In [10]:
# save df to csv
from datetime import datetime
from pathlib import Path

script_dir = Path.cwd()
data_path = script_dir.parent / "data" / "performance"

data_path.mkdir(parents=True, exist_ok=True)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_file = data_path / f"performance_test_{timestamp}.csv"
df.to_csv(data_path / output_file, index=False)

In [11]:
df.sort_values(by="duration_sec", ascending=True, inplace=False)

,modelname,provider,duration_sec,input_tokens,output_tokens,answer
5,claude-haiku-4-5-20251001,anthropic,4.016,16,398,# Pizza Order Transcription\n\n**Pizza Palace ...
3,qwen3:4b-instruct,ollama,7.501,16,246,Sure! Here's a natural and professional transc...
1,llama3.1:8b,ollama,14.183,18,478,Here is the transcription of a pizza order:\n\...
0,gemma3:4b,ollama,14.422,18,704,"Okay, here’s a transcription of a pizza order,..."
2,mistral:7b,ollama,17.175,13,457,"Customer: Hi, I'd like to place an order plea..."
4,qwen3:8b,ollama,26.109,18,935,**Pizza Order Transcription** \n\n**Customer:...


In [12]:
answer_list = df ["answer"].to_list()
for idx, answer in enumerate(answer_list):
    print(f"Answer from model {df['modelname'][idx]} Output tokens: {df['output_tokens'][idx]}:\n {answer}\n")
    print("------------------------------------------------------------\n")

Answer from model gemma3:4b Output tokens: 704:
 Okay, here’s a transcription of a pizza order, aiming for a realistic and detailed conversation:

**Setting:** A phone call to “Tony’s Pizza”

**Characters:**

* **Sarah (Customer):**  Ordering for herself and her roommate.
* **Marco (Pizza Server):** Taking the order.

---

**(Sound of a phone ringing, then being answered)**

**Marco:** Tony’s Pizza, this is Marco speaking, how can I help you tonight?

**Sarah:** Hi Marco, yeah, I’d like to place an order for pickup.

**Marco:** Great!  What can I get for you?

**Sarah:** Okay, let’s start with a large pizza. We’d like a half pepperoni, half mushroom and onion.

**Marco:** Large, half pepperoni, half mushroom and onion. Got it. What kind of crust would you like on that? We have thin, regular, and deep dish.

**Sarah:** Hmm, let’s go with regular crust. Definitely regular.

**Marco:** Okay, regular crust. Anything else on the pizza?

**Sarah:** Yeah, can we add some extra cheese to the w

In [17]:
#to manage results more easely we only want the turn with the order.
#adapt prompt: Only generate the customers turn with the order.
model_name = "qwen3:4b-instruct"
provider = "ollama"
prompt = """
Generate the transcript of a customer who wants to substitute the entire crust, sauce, and cheese for gluten-free, vegan, and lactose-free alternatives, and questions each substitution.
"""


response = generate_example_orders(
    model_name,
    prompt,
    provider,
    temperature
)

print(response.content)
print(len(response.content))


**Customer Service Transcript: Gluten-Free, Vegan, and Lactose-Free Pizza Substitution Request**

---

**[Call Begins – 10:15 AM]**

**Customer:**  
Hi, I’m calling about a pizza order I’m planning. I’d like to know if it’s possible to substitute the entire crust, sauce, and cheese with gluten-free, vegan, and lactose-free alternatives. I’m making this for a friend who has dietary restrictions—gluten sensitivity, vegan lifestyle, and lactose intolerance. So I’m wondering: is that even possible with your current menu?

**Customer Service Representative (CSR):**  
Hi there! Thanks for reaching out. Yes, absolutely—we do offer a fully customizable pizza experience. For your request—gluten-free, vegan, and lactose-free—we can absolutely substitute the crust, sauce, and cheese with certified alternatives. Let me walk you through each component and explain the options.

---

**Customer:**  
Great! So, let’s start with the crust. What’s the gluten-free option?

**CSR:**  
Our gluten-free crus

In [ ]:
# The standard way to ensure consistent output format from LLMs is to use Pydantic models to define and validate the expected structure, 
# which can then be enforced via the API's structured-output feature

from typing import List
from pydantic import BaseModel, Field

class Pizza(BaseModel):
    """Transcription of the pizza order."""

    pizza_order: str = Field(
        ..., description="The pizza order transcription."
    )

class Pizza_order_list(BaseModel):
    """ Generated Pizza orders"""
    # Creates a model so that we can extract multiple entities.
    order_list: List[Pizza]



In [ ]:
from langchain.chat_models import init_chat_model

def generate_structured_example_orders(model_name, prompt, provider=None, temperature = 0.7):
    model = init_chat_model(
        model_name,
        model_provider=provider,
        temperature = temperature
    )
    structured_llm = model.with_structured_output(schema=Pizza_order_list, include_raw=True)
    response = structured_llm.invoke(prompt)
    return response

In [19]:
model_name = "qwen3:4b-instruct"
provider = "ollama"
prompt = """
Generate 5 pizza orders transcribed as if the customer was placing them verbally over the phone. 
Only generate the customers turn with the order.
"""

response = generate_structured_example_orders(
    model_name,
    prompt,
    provider,
    temperature
)
response

NameError: name 'generate_structured_example_orders' is not defined

In [ ]:
#transform pydantic object to Python dict using model_dump()
print(response["raw"].usage_metadata)
print(response["parsed"])
response_dict = response["parsed"].model_dump()
print(type(response_dict))
response_dict

NameError: name 'response' is not defined

In [ ]:
#get the values of the dict 
for order in response_dict['order_list']:
    print(order ['pizza_order'])

In [ ]:
# save  response (pydantic object) as json 
from datetime import datetime
from pathlib import Path


def save_orders_to_file(content) -> Path:
    data_path = Path.cwd().parent / "data" / "example_orders"
    data_path.mkdir(parents=True, exist_ok=True)

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_file = data_path / f"pedidos_pizza_{timestamp}.json"

    output_file.write_text(
        content.model_dump_json(indent=2),
        encoding="utf-8",
    )

    return output_file

In [ ]:
save_orders_to_file(response)

In [ ]:
from pathlib import Path


def load_orders_as_list(filename: str) -> list[str]:
    """Lädt eine JSON-Datei aus data/example_orders und gibt die Orders als Liste zurück."""
    data_path = Path.cwd().parent / "data" / "example_orders"
    path = data_path / filename
    data = json.loads(path.read_text(encoding="utf-8"))

    return [order["pizza_order"] for order in data["order_list"]]

    


In [ ]:
python_list = load_orders_as_list("pedidos_pizza_20260914_115625.json")
print(type(python_list))
python_list
